# Data Dumping

End-to-end walkthrough of the pipeline: load raw data, build the feature matrix, compute PnL labels, then train and validate a model.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')

C_DATA  = '#cce5ff'
C_FEAT  = '#d4edda'
C_TRANS = '#fff3cd'
C_SAMP  = '#e2e3e5'
C_DS    = '#f8d7da'
C_MODEL = '#e2d9f3'

def box(ax, x, y, w, h, label, color, fontsize=9):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                facecolor=color, edgecolor='#444', linewidth=1.2))
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold')

def arrow(ax, x0, y0, x1, y1):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color='#444', lw=1.4))

def dashed_arrow(ax, x0, y0, x1, y1, color='#777'):
    ax.plot([x0, x1], [y0, y1], ls='--', color=color, lw=1.2)
    dx, dy = x1 - x0, y1 - y0
    length = (dx**2 + dy**2) ** 0.5
    f = min(0.04 / length, 0.4)
    ax.annotate('', xy=(x1, y1), xytext=(x1 - dx*f, y1 - dy*f),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

# ----- Boxes -----
box(ax,  0.15, 1.5,  1.45, 2.2,  'Data',                 C_DATA,  fontsize=10)
box(ax,  2.2,  3.75, 2.1,  0.85, 'Feature Blocks',       C_FEAT)
box(ax,  5.1,  3.75, 1.9,  0.85, 'Transforms',           C_TRANS)
box(ax,  7.6,  3.75, 1.9,  0.85, 'Feature\nMatrix X',    C_DS)
box(ax,  2.2,  2.17, 2.1,  0.85, 'Sampler',              C_SAMP)
box(ax,  2.2,  0.75, 2.1,  0.85, 'compute_markout\ncompute_pnl', C_FEAT, fontsize=8)
box(ax,  7.6,  0.75, 1.9,  0.85, 'Labels y\n(pnl_\u03c4)',   C_DS)
box(ax, 10.5,  1.85, 1.8,  1.5,  'train_model\n/\npredict', C_MODEL)

# ----- Solid arrows -----
# Data → three paths
arrow(ax, 1.6, 3.3,  2.2,  4.175)   # → Feature Blocks
arrow(ax, 1.6, 2.6,  2.2,  2.6)     # → Sampler
arrow(ax, 1.6, 1.9,  2.2,  1.175)   # → compute_markout

# Top path: Feature Blocks → Transforms → X
arrow(ax, 4.3,  4.175, 5.1,  4.175)
arrow(ax, 7.0,  4.175, 7.6,  4.175)

# Bottom path: compute_markout → Labels y
arrow(ax, 4.3,  1.175, 7.6,  1.175)

# X and y → Model
arrow(ax, 9.5,  4.175, 10.5, 3.05)
arrow(ax, 9.5,  1.175, 10.5, 2.25)

# ----- Dashed mask connection (Sampler → X and y) -----
# Horizontal dashed line: Sampler right edge → junction at x=8.55
ax.plot([4.3, 8.55], [2.6, 2.6], ls='--', color='#777', lw=1.2)
ax.text(6.4, 2.73, 'mask  [0, 1, 0, 1, ...]', ha='center', va='bottom',
        fontsize=8, color='#555', style='italic')
ax.plot(8.55, 2.6, 'o', color='#777', ms=3.5)   # junction dot

# Vertical dashed: junction → bottom of X
dashed_arrow(ax, 8.55, 2.6, 8.55, 3.75)

# Vertical dashed: junction → top of y
dashed_arrow(ax, 8.55, 2.6, 8.55, 1.6)

ax.text(7, 0.2, 'Batch vectorised  |  O(n log m) per feature group  |  Numba-JIT EWMA',
        ha='center', va='center', fontsize=7.5, color='#666', style='italic')

plt.tight_layout()
plt.show()

## Loading the data

`load_data_with_required_preprocess` reads the raw trades/BBO/liquidation parquet files for a symbol and runs all mandatory preprocessing (casting timestamps to int64 microseconds, lowercasing string columns, sorting by timestamp, and shifting Bybit liquidation timestamps by the known +200ms lag).

Passing `split="sample_train"` restricts the load to one of the small sample date ranges defined in `SPLIT_RANGES`, so this notebook runs against a tiny slice of data instead of the full multi-month dataset.


In [ ]:
trades, bbo, liq_binance, liq_bybit = load_data_with_required_preprocess(
    DATA_DIR, symbol="btcusdt", split="sample_train",
)
trades.shape, bbo.shape, liq_binance.shape, liq_bybit.shape

## Building the feature matrix

`DatasetBuilder` wires together the three independently configurable parts of the pipeline:

- **`features`** — a list of `FeatureBlock`s (`LiqFeatures`, `BookFeatures`, `FlowFeatures`, `VolFeatures`, `TimeFeatures`). Each computes its own columns from the raw frames; swap blocks in/out or change their windows/half-lives independently.
- **`transforms`** — a list of `Transform`s (`DirectionRelativize`, `Winsorize`, `Zscore`, `Clamp`, `FillNanInf`, ...) applied in order to the concatenated feature columns. Each transform can optionally restrict itself to a subset of columns.
- **`sampler`** — a `Sampler` (`EveryTrade`, `VolumeThreshold`, ...) that produces the row mask used to subset the final feature matrix.


In [ ]:
features = [
    LiqFeatures('binance', 'buy',  halflives_s=[1.0, 5.0, 30.0], count_windows_s=[10.0]),
    LiqFeatures('binance', 'sell', halflives_s=[1.0, 5.0, 30.0], count_windows_s=[10.0]),
    LiqFeatures('bybit',   'buy',  halflives_s=[5.0, 30.0]),
    LiqFeatures('bybit',   'sell', halflives_s=[5.0, 30.0]),
    BookFeatures(depth_delta_windows_s=[1.0, 10.0]),
    FlowFeatures(windows_s=[1.0, 5.0, 30.0]),
    VolFeatures(windows_s=[5.0, 60.0], rank_window=1000),
    TimeFeatures(),
]

transforms = [
    DirectionRelativize(),
    Winsorize(0.01),
    Zscore(window=2000, columns=[
        'rolling_vol_5.0s', 'rolling_vol_60.0s', 'spread_bps', 'microprice_dev',
        'same_side_depth', 'opp_side_depth']),
    Clamp(-5.0, 5.0, columns=['rolling_vol_5.0s', 'rolling_vol_60.0s']),
    FillNanInf(),  # always last — cleans up whatever the above left as NaN/inf
]

builder = DatasetBuilder(
    features=features,
    transforms=transforms,
    sampler=VolumeThreshold(500_000),
)

## Using the builder

`builder.build` computes every feature block, applies the transforms, then subsets rows with the sampler mask — returning the feature matrix `X`.


In [ ]:
dataset = builder.build(trades, bbo, liq_binance, liq_bybit)
dataset.head()

## Computing the PnL labels

`compute_markout` looks up the forward mid price at each horizon `tau` for every trade, and `compute_pnl` turns those markouts into maker PnL in bps (`pnl_{tau}`) plus the clipped notional weight `w` used for sample weighting.


In [ ]:
bbo = add_mid(bbo)
trades_labeled = compute_markout(trades, bbo, taus=TAUS)
trades_labeled = compute_pnl(trades_labeled, taus=TAUS)
trades_labeled[["timestamp", "side", "price", "w"] + [f"pnl_{tau}" for tau in TAUS]].head()

## Full pipeline: train and validate

Putting it together: load the train and validation splits, build features and PnL labels for each, fit on train, and check predictions on validation.


In [ ]:
tau = 120

def build_xy(split: str):
    trades, bbo, liq_binance, liq_bybit = load_data_with_required_preprocess(
        DATA_DIR, symbol="btcusdt", split=split,
    )
    X = builder.build(trades, bbo, liq_binance, liq_bybit)

    bbo = add_mid(bbo)
    labeled = compute_markout(trades, bbo, taus=TAUS)
    labeled = compute_pnl(labeled, taus=TAUS)
    labeled = labeled.loc[X.index]

    return X, labeled[f"pnl_{tau}"], labeled["w"]

X_train, y_train, w_train = build_xy("sample_train")
X_val,   y_val,   w_val   = build_xy("sample_val")

model = train_model(X_train, y_train, sample_weight=w_train)
scores_val = predict(model, X_val)

scores_val[:50]